# Exploratory Data Analysis on the Credit Default Dataset

## 1\. Dataset Introduction (from UCI Repository)

This dataset contains information on default payments, demographic factors, credit data, history of payment, and bill statements of credit card clients in Taiwan from April 2005 to September 2005.

  * **Source:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)
  * **Creators:** I-Cheng Yeh, Che-Hui Lien
  * **Citation:** Yeh, I. C., & Lien, C. H. (2009). The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients. Expert Systems with Applications, 36(2), 2473-2480.

### Attribute Information

There are 25 variables:

  * **ID:** ID of each client
  * **LIMIT\_BAL:** Amount of given credit in NT dollars (includes individual and family/supplementary credit)
  * **SEX:** Gender (1=male, 2=female)
  * **EDUCATION:** (1=graduate school, 2=university, 3=high school, 4=others, 5=unknown, 6=unknown)
  * **MARRIAGE:** Marital status (1=married, 2=single, 3=others)
  * **AGE:** Age in years
  * **PAY\_0:** Repayment status in September 2005 (-1=pay duly, 1=payment delay for one month, 2=payment delay for two months, ... 8=payment delay for eight months, 9=payment delay for nine months and above)
  * **PAY\_2 ... PAY\_6:** Repayment status from August 2005 to April 2005 (scale is the same as above)
  * **BILL\_AMT1 ... BILL\_AMT6:** Amount of bill statement from September 2005 to April 2005
  * **PAY\_AMT1 ... PAY\_AMT6:** Amount of previous payment from September 2005 to April 2005
  * **DEFAULT:** Target variable. Default payment next month (1=yes, 0=no)
-----

## 2\. Setup and Data Loading

Let's load the libraries we'll need and the dataset.

-----

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting styles
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
import os

data_file = '/content/credit_default_low.csv'
if not os.path.exists(data_file):
    print(f"Please upload the file...")
    data_file = None
else:
    print(f"'{data_file}' already exists. Skipping upload.")

if data_file and os.path.exists(data_file):
    df_initial = pd.read_csv(data_file)
    print(f"\nSuccessfully loaded data.")
    print(f"Total records in the dataset: {len(df_initial)}")
    print(f"Maximum LIMIT_BAL in the data: {df_initial['LIMIT_BAL'].max()}")
else:
    print("Dataset not loaded. Cannot proceed.")

-----

## 3\. High-Level Data Overview

Let's get a first look at our `df_initial` dataset.

-----

In [ ]:
# 1. Data Types and Non-Null Counts
print("--- Data Info ---")
df_initial.info()

**Questions:**

  * Are there any missing (null) values in your dataset?
  * What are the data types? Are they all appropriate for modeling?
  * How much memory does this dataset use?

-----

In [ ]:
# 2. Check for Duplicates
duplicates = df_initial.duplicated().sum()
print(f"\n--- Duplicates ---")
print(f"Found {duplicates} duplicate row(s).")

**Questions:**

  * What does it mean to have a "duplicate row" in this context?
  * What are the arguments for and against removing these duplicates before training?

-----

In [ ]:
# 3. Summary Statistics for Numerical Features
print("\n--- Numerical Statistics (df_initial) ---")
print(df_initial.describe())

**Questions:**

  * What is the average `AGE` of this customer segment?
  * Look at the `BILL_AMT1` feature. What do the `min` and `25%` values tell you?
  * Compare the `mean` (\$7,388) and `max` (\$149,343) for `PAY_AMT1`. What does this huge difference suggest about payment behavior?

-----

In [ ]:
# 4. Summary for Categorical Features
print("\n--- Categorical Statistics (df_initial) ---")
print("SEX (1=male, 2=female):")
print(df_initial['SEX'].value_counts(normalize=True))

print("\nEDUCATION (1=grad, 2=uni, 3=high school, 4=other):")
print(df_initial['EDUCATION'].value_counts(normalize=True))

print("\nMARRIAGE (1=married, 2=single, 3=other):")
print(df_initial['MARRIAGE'].value_counts(normalize=True))

**Questions:**

  * Which `SEX` is more represented in this low-limit segment?
  * What is the most common `EDUCATION` level?
  * What is the most common `MARRIAGE` status?

-----

## 4\. Univariate Analysis (Analyzing Single Variables)

Let's look at the distribution of each key feature.

### 4.1 Target Variable: `DEFAULT`

This is the most important feature to understand.

-----

In [ ]:
# Set the plot context
sns.set_context("talk")

# Plot
plt.figure(figsize=(8, 6))
ax = sns.countplot(x='DEFAULT', data=df_initial, palette='Reds')
plt.title('Target Variable: Default Distribution (Low-Limit)', fontsize=16)
plt.xlabel('Did they Default? (0 = No, 1 = Yes)', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Add annotations
total = len(df_initial)
for p in ax.patches:
    percentage = f'{(p.get_height() / total):.1%}'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.show()

**Questions:**

  * Is this a "balanced" or "imbalanced" dataset?
  * Why is "accuracy" a bad metric to use for this problem?
  * Which metric(s) (e.g., Precision, Recall, F1-Score, ROC-AUC) would be more appropriate, and why?

-----

### 4.2 Demographics: `AGE`, `SEX`, `EDUCATION`, `MARRIAGE`

-----

In [ ]:
# Reset plot context
sns.set_context("notebook")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plt.suptitle('Distributions of Demographic Features (Low-Limit)', fontsize=20, y=1.02)

# Age
sns.histplot(df_initial['AGE'], kde=True, ax=axes[0, 0], bins=30, color='darkblue')
axes[0, 0].set_title('Age Distribution')

# Sex
sns.countplot(x='SEX', data=df_initial, ax=axes[0, 1], palette='pastel')
axes[0, 1].set_title('Sex Distribution (1=Male, 2=Female)')

# Education
sns.countplot(x='EDUCATION', data=df_initial, ax=axes[1, 0], palette='pastel')
axes[1, 0].set_title('Education (1=Grad, 2=Uni, 3=High, 4=Other)')

# Marriage
sns.countplot(x='MARRIAGE', data=df_initial, ax=axes[1, 1], palette='pastel')
axes[1, 1].set_title('Marriage (1=Married, 2=Single, 3=Other)')

plt.tight_layout()
plt.show()

**Questions:**

  * What is the primary age group for this low-limit segment?
  * Does the `SEX` distribution (more females) surprise you? Why or why not?
  * University and High School are the dominant `EDUCATION` levels. What might this imply about this customer segment?

-----

### 4.3 Payment History: `PAY_0` (Status last month)

`PAY_0` is the repayment status from last month (Sept 2005). This is likely a very predictive feature.

-----

In [ ]:
plt.figure(figsize=(12, 7))
sns.countplot(x='PAY_0', data=df_initial, palette='coolwarm')
plt.title('Repayment Status in Sept 2005 (PAY_0)')
plt.xlabel('Payment Status (0 = Not Delayed, 1 = 1mo delay, etc.)')
plt.ylabel('Count')
plt.show()

**Questions:**

  * What is the *vast* majority of repayment statuses? What does our cleaning (grouping -2, -1, 0 into 0) tell you?
  * Are there many customers who have delayed for 2 or more months?
  * How do you think `PAY_0` will relate to the `DEFAULT` target?

-----

## 5\. Bivariate Analysis (Feature vs. Target)

Now let's see how our features relate to `DEFAULT`.

### 5.1 Demographics vs. `DEFAULT`

-----

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
plt.suptitle('Demographics vs. Default Status (Low-Limit)', fontsize=18, y=1.03)

# Sex vs Default
sns.countplot(x='SEX', hue='DEFAULT', data=df_initial, ax=axes[0], palette='muted')
axes[0].set_title('Default by Sex (1=Male, 2=Female)')

# Marriage vs Default
sns.countplot(x='MARRIAGE', hue='DEFAULT', data=df_initial, ax=axes[1], palette='muted')
axes[1].set_title('Default by Marriage (1=Married, 2=Single, 3=Other)')

# Education vs Default
sns.countplot(x='EDUCATION', hue='DEFAULT', data=df_initial, ax=axes[2], palette='muted')
axes[2].set_title('Default by Education (1=Grad, 2=Uni, ...)')

plt.tight_layout()
plt.show()

**Questions:**

  * Look at the "Default by Sex" plot. Does one gender appear to have a *higher rate* of default? (Hint: compare the *proportion* of red-to-blue for each bar).
  * Which `MARRIAGE` status seems to have the highest default rate?
  * What about `EDUCATION`? Does a higher education level seem to correspond to a lower default rate?

-----

### 5.2 Payment History vs. `DEFAULT`

This is the most important relationship. Let's look at `PAY_0` vs `DEFAULT`.

-----

In [ ]:
# Calculate default rate by PAY_0 status
pay_0_crosstab = pd.crosstab(df_initial['PAY_0'], df_initial['DEFAULT'], normalize='index') * 100
pay_0_crosstab = pay_0_crosstab.rename(columns={0: 'No Default %', 1: 'Default %'})
print(pay_0_crosstab)

# Plot it
pay_0_crosstab['Default %'].plot(kind='bar', figsize=(12, 7), color='salmon')
plt.title('Default Rate (%) by Last Month\'s Payment Status (PAY_0)')
plt.xlabel('Payment Status (0 = Not Delayed, 1 = 1mo delay, ...)')
plt.ylabel('Default Rate (%)')
plt.xticks(rotation=0)
plt.show()

**Questions:**

  * What happens to the default rate when a customer is *not* delayed (`PAY_0 = 0`)?
  * What happens to the default rate as soon as a customer is delayed by even one month (`PAY_0 = 1`)?
  * What happens after 2 months of delay (`PAY_0 = 2`)?
  * Based *only* on this plot, how important is this feature for your model?

-----

## 6\. Multivariate Analysis & Correlations

Let's look at all the numerical features at once.

-----

In [ ]:
# We will focus on the most relevant numerical columns for the heatmap
corr_cols = [
    'LIMIT_BAL', 'AGE', 'DEFAULT',
    'PAY_0', 'PAY_2', 'PAY_3',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3'
]

# Calculate the correlation matrix
corr_matrix = df_initial[corr_cols].corr()

# Plot the heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap (Low-Limit Segment)', fontsize=18)
plt.show()

**Questions:**

  * Look at the `DEFAULT` row/column. What is the single strongest positive predictor of `DEFAULT`?
  * What are the `BILL_AMT` features (1-3) highly correlated with? What is this multicollinearity, and why might it be a problem for *some* model types (like Linear Regression)?
  * What are the `PAY_X` features correlated with? Does this make sense?
  * Is `AGE` strongly correlated with `DEFAULT`?


-----

## 7\. Key Questions for Your MLOps Project

This EDA is your foundation. As you build your model, you must consider the "Ops" (Operations) part of MLOps. Here are the key questions your team needs to think about.

### Data Governance & Quality

  * **Data Provenance:** Where did this data come from? Is it from a single bank? All of Taiwan? What year? The documentation says 2005—how might customer behavior have changed since then?
  * **Data Cleaning:** We grouped several `EDUCATION` and `MARRIAGE` categories. Was this the right choice? What information might we have lost? What about the `PAY_X` columns?
  * **PII (Personally Identifiable Information):** This data is *supposedly* anonymous. But could `AGE`, `SEX`, `EDUCATION`, and `MARRIAGE` be combined to re-identify someone? What are our responsibilities?

### Responsible AI (Fairness & Bias)

  * **Protected Attributes:** Your model will use `AGE` and `SEX`. You *must* check your final model for fairness. Does it have a higher error rate (e.g., more False Negatives) for one gender over the other? Or for younger vs. older people?
  * **Segment Bias:** Your *entire training set* consists of "low-limit" customers. This is a massive **sampling bias**. Your model will learn *only* the behaviors of this group. What do you think will happen if the bank tries to use your model on a "high-limit" customer?
  * **Feedback Loops:** If your model *denies* credit (a "False Positive") to someone, they won't become a customer. You will never get new data on whether they *would* have defaulted. How can your model learn from its mistakes if it creates this kind of feedback loop?

### Model Decay & MLOps Strategy

  * **What is "Data Drift"?** Data drift is when the *input data* in production looks different from the training data. For example, what if a new marketing campaign signs up thousands of new customers who are all "single" (`MARRIAGE=2`)? Your model, trained on mostly "married" people, might not perform well.
  * **What is "Concept Drift"?** Concept drift is when the *relationship* between features and the target changes. What if a new law passes that provides debt relief, and suddenly people with `PAY_0 = 2` (2-month delay) stop defaulting? Your model's most important rule (`PAY_0=2 -> high risk`) is now wrong.
  * **The "High-Limit" Problem (Your Core Task):** You know your model is only trained on "low-limit" data. Your primary MLOps challenge will be to detect what happens when "high-limit" data shows up.
      * How will you **monitor** the `LIMIT_BAL` of incoming data?
      * What **alert** would you set up (e.g., "Alert me if `LIMIT_BAL` \> 150,000")?
  * **Retraining Strategy:**
      * When you detect this performance drop, what's your plan?
      * Do you retrain your *current* model on the new data?
      * Do you build a *second* model just for high-limit customers?
      * Do you build one "global" model on *all* the data? What are the pros and cons of each?